# Pipeline Helper
Utilities for working with `analysis.json` files: reading, inspecting scenes/subsegments, joining LLM evaluations, and generating VLM prompts.

In [ ]:
import json
from pathlib import Path
from typing import Any

## Config

In [ ]:
ANALYSIS_PATH = Path("../local/videos/DJI_20250514105245_0135_D.analysis.json")

## I/O

In [ ]:
def read_analysis(path: str | Path) -> dict:
    """Load an analysis.json file and return the parsed dict."""
    with open(path) as f:
        return json.load(f)

## Scene attribute extraction

In [ ]:
DEFAULT_SCENE_ATTRS = ["start_time", "end_time"]


def _get_nested(obj: dict, key: str) -> Any:
    """Resolve a dotted key path like 'channel_metrics.pan.rms' from a dict."""
    for part in key.split("."):
        if not isinstance(obj, dict) or part not in obj:
            return None
        obj = obj[part]
    return obj


def extract_scene_attributes(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
) -> list[dict]:
    """
    Extract specified attributes from each scene.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    attributes:
        Scene-level keys to include.  Dotted paths are supported,
        e.g. ``["start_time", "end_time", "quality_score"]``.

    Returns
    -------
    List of dicts, one per scene, always including ``scene_id``.
    """
    rows = []
    for scene in analysis.get("scenes", []):
        row = {"scene_id": scene["scene_id"]}
        for attr in attributes:
            row[attr] = _get_nested(scene, attr)
        rows.append(row)
    return rows


def print_scenes_json(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
    indent: int = 2,
) -> None:
    """Print per-scene attributes as a JSON array."""
    rows = extract_scene_attributes(analysis, attributes)
    print(json.dumps(rows, indent=indent))


def print_scenes_table(
    analysis: dict,
    attributes: list[str] = DEFAULT_SCENE_ATTRS,
    float_fmt: str = ".3f",
) -> None:
    """Print per-scene attributes as a plain-text table."""
    rows = extract_scene_attributes(analysis, attributes)
    if not rows:
        print("(no scenes)")
        return

    columns = list(rows[0].keys())

    # Format all values as strings
    str_rows = []
    for row in rows:
        str_row = {}
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                str_row[col] = format(val, float_fmt)
            else:
                str_row[col] = str(val) if val is not None else ""
        str_rows.append(str_row)

    # Column widths
    widths = {col: max(len(col), *(len(r[col]) for r in str_rows)) for col in columns}

    sep = "+" + "+".join("-" * (widths[c] + 2) for c in columns) + "+"
    header = "|" + "|".join(f" {c.ljust(widths[c])} " for c in columns) + "|"

    print(sep)
    print(header)
    print(sep)
    for row in str_rows:
        line = "|" + "|".join(f" {row[c].ljust(widths[c])} " for c in columns) + "|"
        print(line)
    print(sep)

### Demo – scenes

In [ ]:
analysis = read_analysis(ANALYSIS_PATH)

print_scenes_json(analysis)
print()
print_scenes_table(analysis)

## Subsegment attribute extraction

In [ ]:
DEFAULT_SUBSEG_ATTRS = ["start_time", "end_time"]


def extract_subsegment_attributes(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
) -> list[dict]:
    """
    Extract specified attributes from every subsegment across all scenes.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    attributes:
        Subsegment-level keys to include.  Dotted paths are supported,
        e.g. ``["start_time", "end_time", "channel_metrics.pan.rms"]``.

    Returns
    -------
    List of dicts, one per subsegment, always including ``scene_id``
    and ``subsegment_id``.
    """
    rows = []
    for scene in analysis.get("scenes", []):
        for sub in scene.get("subsegments", []):
            row = {
                "scene_id": scene["scene_id"],
                "subsegment_id": sub["subsegment_id"],
            }
            for attr in attributes:
                row[attr] = _get_nested(sub, attr)
            rows.append(row)
    return rows


def print_subsegments_json(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
    indent: int = 2,
) -> None:
    """Print per-subsegment attributes as a JSON array."""
    rows = extract_subsegment_attributes(analysis, attributes)
    print(json.dumps(rows, indent=indent))


def print_subsegments_table(
    analysis: dict,
    attributes: list[str] = DEFAULT_SUBSEG_ATTRS,
    float_fmt: str = ".3f",
) -> None:
    """Print per-subsegment attributes as a plain-text table."""
    rows = extract_subsegment_attributes(analysis, attributes)
    if not rows:
        print("(no subsegments)")
        return

    columns = list(rows[0].keys())

    str_rows = []
    for row in rows:
        str_row = {}
        for col in columns:
            val = row[col]
            if isinstance(val, float):
                str_row[col] = format(val, float_fmt)
            else:
                str_row[col] = str(val) if val is not None else ""
        str_rows.append(str_row)

    widths = {col: max(len(col), *(len(r[col]) for r in str_rows)) for col in columns}

    sep = "+" + "+".join("-" * (widths[c] + 2) for c in columns) + "+"
    header = "|" + "|".join(f" {c.ljust(widths[c])} " for c in columns) + "|"

    print(sep)
    print(header)
    print(sep)
    for row in str_rows:
        line = "|" + "|".join(f" {row[c].ljust(widths[c])} " for c in columns) + "|"
        print(line)
    print(sep)

### Demo – subsegments

In [ ]:
print_subsegments_json(analysis)
print()
print_subsegments_table(analysis)

## Join LLM evaluation

In [ ]:
def join_llm_evaluation(
    analysis: dict,
    llm_json: str | Path | list[dict],
    join_key: str = "scene_id",
    field_prefix: str = "llm_",
) -> dict:
    """
    Merge LLM evaluation results into the analysis dict (in-place copy).

    The LLM JSON must be a list of objects each containing ``join_key``
    (default ``scene_id``).  All other fields from each LLM object are
    injected into the matching scene under the given ``field_prefix``.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.  Not mutated; a shallow-merged copy
        is returned.
    llm_json:
        Path to a JSON file **or** an already-parsed list of dicts.
    join_key:
        The field used to match LLM records to scenes (default ``scene_id``).
    field_prefix:
        Prefix added to every field imported from the LLM evaluation
        to avoid name collisions (default ``llm_``).

    Returns
    -------
    A new analysis dict with LLM fields merged into each scene.
    """
    if not isinstance(llm_json, list):
        with open(llm_json) as f:
            llm_records = json.load(f)
    else:
        llm_records = llm_json

    llm_by_key = {rec[join_key]: rec for rec in llm_records if join_key in rec}

    merged_scenes = []
    for scene in analysis.get("scenes", []):
        scene_copy = dict(scene)
        key_val = scene.get(join_key)
        if key_val in llm_by_key:
            for k, v in llm_by_key[key_val].items():
                if k != join_key:
                    scene_copy[f"{field_prefix}{k}"] = v
        merged_scenes.append(scene_copy)

    return {**analysis, "scenes": merged_scenes}

### Demo – join LLM evaluation

In [ ]:
# Example: provide a list of dicts directly (or pass a path to a JSON file)
example_llm_eval = [
    {"scene_id": 0, "description": "Slow pan over landscape", "keep": True, "rating": 4},
    {"scene_id": 1, "description": "Fast zoom in",            "keep": False, "rating": 2},
]

merged = join_llm_evaluation(analysis, example_llm_eval)

# Verify the new fields appear in the first two scenes
for scene in merged["scenes"][:3]:
    llm_fields = {k: v for k, v in scene.items() if k.startswith("llm_")}
    print(f"scene {scene['scene_id']}: {llm_fields}")

## Extract analysis subset (strip per-frame data)

In [ ]:
# Top-level keys that contain per-frame arrays – excluded from the subset
_PER_FRAME_KEYS = {"frame_metrics", "flow_stats", "flow_decomposition"}


def extract_analysis_subset(
    analysis: dict,
    exclude_keys: set[str] | None = None,
    include_subsegments: bool = True,
) -> dict:
    """
    Return a lightweight copy of the analysis dict without per-frame arrays.

    Keeps all top-level metadata (video_path, duration, num_scenes, boundaries…)
    and the full scene list including subsegments.  Per-frame arrays
    (frame_metrics, flow_stats, flow_decomposition) are dropped by default.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    exclude_keys:
        Override the set of top-level keys to drop.  Defaults to
        ``{"frame_metrics", "flow_stats", "flow_decomposition"}``.
    include_subsegments:
        When ``False``, the ``subsegments`` list is removed from each scene
        (useful for an even more compact summary).

    Returns
    -------
    A new dict with the requested keys removed.
    """
    excluded = exclude_keys if exclude_keys is not None else _PER_FRAME_KEYS

    subset = {k: v for k, v in analysis.items() if k not in excluded}

    if not include_subsegments and "scenes" in subset:
        trimmed_scenes = []
        for scene in subset["scenes"]:
            scene_copy = {k: v for k, v in scene.items() if k != "subsegments"}
            trimmed_scenes.append(scene_copy)
        subset["scenes"] = trimmed_scenes

    return subset

### Demo – subset

In [ ]:
subset = extract_analysis_subset(analysis)
print("Top-level keys kept:", list(subset.keys()))
print(f"Scenes: {len(subset['scenes'])}, first scene keys: {list(subset['scenes'][0].keys())}")

## VLM prompt generation

In [ ]:
def _format_time(seconds: float) -> str:
    """Format seconds as MM:SS.mmm for human-readable timestamps."""
    minutes = int(seconds) // 60
    secs = seconds - minutes * 60
    return f"{minutes:02d}:{secs:06.3f}"


def generate_vlm_prompt(
    analysis: dict,
    template: str | None = None,
    time_format: str = "seconds",
) -> str:
    """
    Build a VLM prompt that includes the pre-computed scene segmentation.

    The scene boundaries are embedded directly so the VLM does **not** need
    to perform its own segmentation – it should evaluate the provided scenes
    as-is.

    Parameters
    ----------
    analysis:
        Parsed analysis.json dict.
    template:
        Optional prompt template string.  Use ``{scenes}`` as the placeholder
        for the scene block and ``{num_scenes}`` for the scene count.
        If ``None``, a minimal placeholder template is used so you can see
        the scene block output and slot in the real template later.
    time_format:
        ``"seconds"`` (default) or ``"timecode"`` (MM:SS.mmm).

    Returns
    -------
    The complete prompt string.
    """
    scenes = analysis.get("scenes", [])

    lines = []
    for scene in scenes:
        sid = scene["scene_id"]
        start = scene["start_time"]
        end = scene["end_time"]

        if time_format == "timecode":
            start_str = _format_time(start)
            end_str = _format_time(end)
        else:
            start_str = f"{start:.3f}s"
            end_str = f"{end:.3f}s"

        lines.append(f"  Scene {sid}: {start_str} – {end_str}")

    scene_block = "\n".join(lines)

    if template is None:
        template = (
            "[PROMPT TEMPLATE PLACEHOLDER]\n"
            "\n"
            "The video has been pre-segmented into {num_scenes} scenes.\n"
            "Do NOT perform your own segmentation. Evaluate each scene as provided:\n"
            "\n"
            "{scenes}"
        )

    return template.format(scenes=scene_block, num_scenes=len(scenes))

### Demo – VLM prompt

In [ ]:
prompt = generate_vlm_prompt(analysis, time_format="timecode")
print(prompt)

In [ ]:
# With a custom template (fill in the real one when ready)
custom_template = """\
You are evaluating drone footage. For each scene listed below, provide:
- A short description of camera movement and subject
- A quality rating from 1-5
- Whether the scene should be kept (yes/no) and why

The video contains {num_scenes} pre-segmented scenes:
{scenes}

Respond with a JSON array with one object per scene.
"""

print(generate_vlm_prompt(analysis, template=custom_template, time_format="seconds"))